# Co-clustering Sliced GW Demo

This notebook loads two CAPOD  meshes, computes low-rank GW co-clusters, selects anchors with two strategies, and builds anchored 1D Wasserstein matchings using the refactored Python modules.

## 1. Setup

We define the data paths, reproducibility settings, and algorithm parameters used throughout the demo.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

source_root = PROJECT_ROOT
data_root = source_root / 'data' / 'CAPOD'

n = 5000
rank = 10
reg = 5e-5
shape_class = 2
shape_sample_1 = 1
shape_sample_2 = 3
seed = 0
n_neighbors = 10
min_component_size = 1
chunk_size = 512
gw_eval_chunk_size = 64



## 2. Imports

The notebook only imports helper modules and a small set of runtime libraries.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ot
import trimesh
from sklearn.metrics import pairwise_distances

from co_cluster import compute_lowrank_gw_coclusters, plot_coclusters_3d
from anchor_selection import compute_anchors, plot_anchors_3d, plot_clusters_with_representatives_pretty
from anchored_1d_wasserstein import (
    anchored_sliced_gw_match,
    gw_value_from_distance_matrices,
    hard_match_from_coupling,
    plot_final_alignment,
    print_coupling_diagnostics,
)

## 3. Load CAPOD horse point clouds

We reconstruct the two source point clouds by loading the original `.obj` meshes and sampling `n` points from each mesh.

In [ ]:
def capod_obj_path(data_root: Path, shape_class: int, shape_sample: int) -> Path:
    sample_id = (shape_class - 1) * 12 + shape_sample
    path = data_root / f'class{shape_class}' / f'm{sample_id}.obj'
    if not path.exists():
        raise FileNotFoundError(
            f'Missing CAPOD mesh: {path}. Please check data_root={data_root} and the CAPOD files.'
        )
    return path

def load_capod_point_cloud(mesh_path: Path, n_points: int, seed: int) -> np.ndarray:
    mesh = trimesh.load(mesh_path, force='mesh')
    np.random.seed(seed)
    X = mesh.sample(n_points)
    return np.asarray(X, dtype=float)

path1 = capod_obj_path(data_root, shape_class, shape_sample_1)
path2 = capod_obj_path(data_root, shape_class, shape_sample_2)

X1 = load_capod_point_cloud(path1, n, seed=seed)
X2 = load_capod_point_cloud(path2, n, seed=seed + 1)

a = ot.unif(len(X1))
b = ot.unif(len(X2))

print('X1 shape:', X1.shape)
print('X2 shape:', X2.shape)

## 4. Compute low-rank GW co-clusters

We reuse POT's `lowrank_gromov_wasserstein_samples`, reconstruct the low-rank coupling, and visualize the hard co-cluster assignments.

In [ ]:
co_out = compute_lowrank_gw_coclusters(
    X1,
    X2,
    rank=rank,
    reg=reg,
    numItermax=n,
    stopThr=1e-9,
    a=a,
    b=b,
    random_state=seed,
)

Q = co_out['Q']
R = co_out['R']
g = co_out['g']
P_lowrank = co_out['P_lowrank']
label1 = co_out['label1']
label2 = co_out['label2']

print('Q shape:', Q.shape)
print('R shape:', R.shape)
print('g shape:', g.shape)
print('P_lowrank shape:', P_lowrank.shape)
print('P_lowrank mass:', float(P_lowrank.sum()))

_ = plot_coclusters_3d(X1, X2, label1, label2, rank=rank)
plt.show()

## 5. Compute anchors with two methods

We test both centroid anchors and connected-component medoid anchors for the two point clouds.

In [ ]:
centroid_out_X = compute_anchors(X1, label1, rank=rank, method='centroid')
centroid_out_Y = compute_anchors(X2, label2, rank=rank, method='centroid')

medoid_out_X = compute_anchors(
    X1,
    label1,
    rank=rank,
    method='component_medoid',
    n_neighbors=n_neighbors,
    min_component_size=min_component_size,
    chunk_size=chunk_size,
    verbose=True,
)
medoid_out_Y = compute_anchors(
    X2,
    label2,
    rank=rank,
    method='component_medoid',
    n_neighbors=n_neighbors,
    min_component_size=min_component_size,
    chunk_size=chunk_size,
    verbose=True,
)

print('Centroid anchors X shape:', centroid_out_X['anchors'].shape)
print('Centroid anchors Y shape:', centroid_out_Y['anchors'].shape)
print('Medoid anchors X shape:', medoid_out_X['anchors'].shape)
print('Medoid anchors Y shape:', medoid_out_Y['anchors'].shape)

In [ ]:
plot_anchors_3d(
    X1,
    label1,
    centroid_out_X['anchors'],
    rank=rank,
    title='View 1 centroid anchors',
    point_size=10,
    point_opacity=0.4,
    representative_scale=0.05,
    show_labels=False,
)

plot_anchors_3d(
    X2,
    label2,
    centroid_out_Y['anchors'],
    rank=rank,
    title='View 2 centroid anchors',
    point_size=10,
    point_opacity=0.4,
    representative_scale=0.05,
    show_labels=False,
)

plot_clusters_with_representatives_pretty(
    X1,
    label1,
    medoid_out_X['component_info'],
    title='View 1 medoid anchors',
    point_size=10,
    point_opacity=0.4,
    representative_scale=0.05,
    show_labels=False,
)

plot_clusters_with_representatives_pretty(
    X2,
    label2,
    medoid_out_Y['component_info'],
    title='View 2 medoid anchors',
    point_size=10,
    point_opacity=0.4,
    representative_scale=0.05,
    show_labels=False,
)

## 6. Build anchored 1D Wasserstein couplings

We assemble a final sparse coupling for both anchor strategies and inspect the resulting marginals.

In [ ]:
centroid_match = anchored_sliced_gw_match(
    X1=X1,
    X2=X2,
    label1=label1,
    label2=label2,
    anchors_X=centroid_out_X['anchors'],
    anchors_Y=centroid_out_Y['anchors'],
    rank=rank,
    a=a,
    b=b,
    anchor_mode='all',
    block_mass='average',
    return_dense=False,
    verbose=True,
)

medoid_match = anchored_sliced_gw_match(
    X1=X1,
    X2=X2,
    label1=label1,
    label2=label2,
    anchors_X=medoid_out_X['anchors'],
    anchors_Y=medoid_out_Y['anchors'],
    rank=rank,
    a=a,
    b=b,
    anchor_mode='all',
    block_mass='average',
    return_dense=False,
    verbose=True,
)


## 7. Hard matching and final alignment

We convert each final coupling to an argmax hard matching and render paper-style final alignment figures for the low-rank, centroid-anchor, and component-medoid couplings.

In [ ]:
hard_match_centroid = hard_match_from_coupling(centroid_match['P_sliced_sparse'], direction='X1_to_X2')
hard_match_medoid = hard_match_from_coupling(medoid_match['P_sliced_sparse'], direction='X1_to_X2')

print('Hard match centroid shape:', hard_match_centroid.shape)
print('Hard match medoid shape:', hard_match_medoid.shape)
print('First 10 centroid matches:', hard_match_centroid[:10])
print('First 10 medoid matches:', hard_match_medoid[:10])



rows_medoid, cols_medoid, fig_medoid, ax_medoid = plot_final_alignment(
    X1,
    X2,
    medoid_match['P_sliced_sparse'],
    n_lines=500,
    candidate_pool=2000,
    pair_mode='argmax_X1_to_X2',
    max_points=1800,
    vertical_sep=1.5,
    point_size=3,
    point_alpha=0.7,
    line_width=0.6,
    line_alpha=0.18,
    line_color='#aaaaaa',
    top_color='#d88c8c',
    bottom_color='#6b6fc9',
    save_path='alignment_paper_final.png',
    seed=42,
)

Dist1_full = pairwise_distances(X1, X1, metric='euclidean')
Dist2_full = pairwise_distances(X2, X2, metric='euclidean')

gw_medoid = gw_value_from_distance_matrices(Dist1_full, Dist2_full, medoid_match['P_sliced_sparse'], normalize_mass=True, chunk_size=gw_eval_chunk_size)
